# Multi-Compartment Homogeneous Fleet Vehicle Routing Problem (MIP)

In [1]:
import DataFrames, Plots, SparseArrays
using JuMP, HiGHS, CPLEX, LinearAlgebra, DataFrames

In [ ]:
function parameter_data()
    K =  15        # Number of Trucks
    P = 3         # Number of Products
    
    Vc = 4          # Number of Customer 
    d = rand(5:22, Vc+1, Vc+1)
    for i in 1:Vc+1
        d[i, i] =0
    end
    d[end, :] .= d[:, end]
    
    Qp = [100, 100, 100]        # Compartment Capacity
    Qmax = sum(Qp) * ones(1, K)    # Maximum Capacity of each truck
    q = rand(5:20,Vc,P)         # Demand of the customers

    L = 2500                    # Maximum length a route can be covered by a truck
    return Vc, K, P, d, Qmax, q, Qp, L
end
Vc, K, P, d, Qmax, q, Qp, L = parameter_data()

In [ ]:
model = Model(CPLEX.Optimizer)
set_silent(model)
@variable(model, x[1:Vc+1, 1:Vc+1, 1:K] >= 0, Bin)
@variable(model, z[1:Vc, 1:K, 1:P] >= 0, Bin)
@variable(model, y[1:Vc+1, 1:K], Bin)
@variable(model, u[1:Vc, 1:K, 1:P])
@objective(model, Min, sum(d[i, j] * x[i, j, k] for i in 1:(Vc+1) for j in 1:(Vc+1) for k in 1:K if i != j))
@constraint(model, [i in 1:Vc], sum(y[i, :]) == 1)
@constraint(model, sum(y[Vc+1, :]) <= K)
@constraint(model, [j in 1:Vc+1, k in 1:K], sum(x[i, j, k] for i in 1:(Vc+1) if i!=j) == y[j, k])
@constraint(model, [i in 1:Vc+1, k in 1:K], sum(x[i, j, k] for j in 1:(Vc+1) if j!=i) == y[i, k])
@constraint(model, [k in 1:K], sum(q[i, p] * y[i, k] for i in 1:Vc for p in P) <= Qmax[k])
@constraint(model, [j in 1:Vc, k in 1:K, p in 1:P], z[j, k, p] <= sum(x[i, j, k] for i in 1:Vc+1 if i!=j))
@constraint(model, [j in 1:Vc, p in 1:P], sum(z[j, :, p]) == 1)
@constraint(model, [k in 1:K, p in 1:P], sum(z[j, k, p] * q[j, p] for j in 1:Vc) <= Qp[p])
@constraint(model, [k in 1:K], sum(d[i, j] * x[i, j, k] for i in 1:(Vc+1) for j in 1:(Vc+1) if i != j) <= L)
@constraint(model, [i in 1:Vc, j in 1:Vc, j!=i, k in 1:K, p in 1:P], u[i, k, p] - u[j, k, p] + Qp[p] * x[i, j, k] <= Qp[p] - q[i, p])
@constraint(model, [i in 1:Vc, k in 1:K, p in 1:P], q[i, p] - u[i, k, p] <= 0)
@constraint(model, [i in 1:Vc, k in 1:K, p in 1:P], Qp[p] - u[i, k, p] >= 0)
set_time_limit_sec(model, 10.0)
optimize!(model)
xVals = value.(x)
for k = 1:size(xVals, 3), i = 1:size(xVals, 1), j = 1:size(xVals, 2)
    if xVals[i, j, k] > 0
        println("x($i, $j, $k): ", xVals[i, j, k])
    end
end
termination_status(model)

In [ ]:
using Plots

loc_x = rand(Vc+1)    # x-coordinates
loc_y = rand(Vc+1)    # y-coordinates

# Create a scatter plot
scatter(loc_x[1:end-1], loc_y[1:end-1], color=:yellow, marker=:star, aspect_ratio=1, legend=:outertopright, label="Customers")

# Add the depot point to the plot
scatter!([loc_x[end]], [loc_y[end]], color=:red, marker=:diamond, markersize=6, label="Depot")

for i in 1:Vc+1, j in 1:Vc+1, k in 1:K
    if xVals[i, j, k] > 0.5
        plot!([loc_x[i], loc_x[j]], [loc_y[i], loc_y[j]], line=:path, color=:blue, alpha=0.5, label="")
    end
end

# Display the plot
display(plot!())  # Assuming "figure" is your plot


In [2]:
mutable struct Data
    d::Matrix{Float64}
    q::Matrix{Int64}
    Qp::Vector{Int64}
    Vc::Int64
    P::Int64
    L::Int64
    # initializer
    function Data(d::Matrix{Int64}, q::Matrix{Int64}, Qp::Vector{Int64}, P, L)
        # number of customers
        Vc = size(q,1)-1 
        new(d, q, Qp, Vc, P, L)
    end

end

In [3]:
A = [1	3	4	3	10
2	6	8	6	20
3	6	8	6	20
4	9	12	9	30
5	3	4	3	10
6	6	8	6	20
7	9	12	9	30
8	9	12	9	30
9	3	4	3	10
10	6	8	6	20
11	3	4	3	10
12	6	8	6	20
13	6	8	6	20
14	9	12	9	30
15	3	4	3	10
16	0	0	0	0
]
P = 3
q = A[:, 2:4] # demand
Vc = size(q, 1)-1 
G = zeros(Int,Vc)
for i in 1:Vc
    G[i] = sum(q[i, p] for p in 1:P)
end

d = rand(5:22, Vc+1, Vc+1)
    for i in 1:Vc+1
        d[i, i] =0
    end
d[end, :] .= d[:, end]
Qp = [40, 40, 40]       # Compartment Capacity 
L = 2500
data = Data(d, q, Qp, P, L)

Data([0.0 16.0 … 7.0 22.0; 17.0 0.0 … 14.0 7.0; … ; 19.0 18.0 … 0.0 15.0; 22.0 7.0 … 15.0 0.0], [3 4 3; 6 8 6; … ; 3 4 3; 0 0 0], [40, 40, 40], 15, 3, 2500)

In [4]:
mutable struct NaiveMIP
    model::Model
    K::Int64
    x::Array{VariableRef, 3}
    y::Matrix{VariableRef}
    z::Array{VariableRef, 3}
    u::Array{VariableRef, 3}

    # initializer
    function NaiveMIP(data::Data)
        
        d = data.d
        q = data.q
        Vc = data.Vc
        P = data.P
        L = data.L
        K = Vc
        Qp = data.Qp
        Qmax = sum(Qp) * ones(1, K)

        model = Model(CPLEX.Optimizer) # using HiGHS optimizer to avoid cplex 1217 error
        @variable(model, x[1:Vc+1, 1:Vc+1, 1:K] >= 0, Bin)
        @variable(model, z[1:Vc, 1:K, 1:P] >= 0, Bin)
        @variable(model, y[1:Vc+1, 1:K], Bin)
        @variable(model, u[1:Vc, 1:K, 1:P])

        # Statement 1: Add constraints to `model`
        
        @constraint(model, [i in 1:Vc], sum(y[i, :]) == 1)
        @constraint(model, sum(y[Vc+1, :]) <= K)
        @constraint(model, [j in 1:Vc+1, k in 1:K], sum(x[i, j, k] for i in 1:(Vc+1) if i!=j) == y[j, k])
        @constraint(model, [i in 1:Vc+1, k in 1:K], sum(x[i, j, k] for j in 1:(Vc+1) if j!=i) == y[i, k])
        @constraint(model, tr_cp[k in 1:K], sum(G[i] * y[i, k] for i in 1:Vc) <= Qmax[k])
        @constraint(model, [j in 1:Vc, k in 1:K, p in 1:P], z[j, k, p] <= sum(x[i, j, k] for i in 1:Vc+1 if i!=j))
        @constraint(model, [j in 1:Vc, p in 1:P], sum(z[j, :, p]) == 1)
        @constraint(model, [k in 1:K, p in 1:P], sum(z[j, k, p] * q[j, p] for j in 1:Vc) <= Qp[p])
        @constraint(model, [k in 1:K], sum(d[i, j] * x[i, j, k] for i in 1:(Vc+1) for j in 1:(Vc+1) if i != j) <= L)
        @constraint(model, [i in 1:Vc, j in 1:Vc, j!=i, k in 1:K, p in 1:P], u[i, k, p] - u[j, k, p] + Qp[p] * x[i, j, k] <= Qp[p] - q[i, p])
        @constraint(model, [i in 1:Vc, k in 1:K, p in 1:P], q[i, p] - u[i, k, p] <= 0)
        @constraint(model, [i in 1:Vc, k in 1:K, p in 1:P], Qp[p] - u[i, k, p] >= 0)
        
        # Statement 2: Add objective to `model`
        @objective(model, Min, sum(d[i, j] * x[i, j, k] for i in 1:(Vc+1) for j in 1:(Vc+1) for k in 1:K if i != j))
    
        new(model, K, x, y, z, u)
    end
end

In [14]:
nmip = NaiveMIP(data)
set_time_limit_sec(nmip.model, 400.0)
optimize!(nmip.model)
xVals = value.(nmip.x)
for k = 1:size(xVals, 3), i = 1:size(xVals, 1), j = 1:size(xVals, 2)
    if xVals[i, j, k] > 0
        println("x($i, $j, $k): ", xVals[i, j, k])
    end
end

Version identifier: 22.1.1.0 | 2022-11-27 | 9160aff4d
CPXPARAM_TimeLimit                               400
Tried aggregator 1 time.
MIP Presolve eliminated 2041 rows and 240 columns.
MIP Presolve modified 6750 coefficients.
Reduced MIP has 10725 rows, 5190 columns, and 48630 nonzeros.
Reduced MIP has 4515 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.06 sec. (27.33 ticks)
Found incumbent of value 311.000000 after 0.12 sec. (62.80 ticks)
Probing time = 0.64 sec. (112.06 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 765 rows and 675 columns.
Reduced MIP has 9960 rows, 4515 columns, and 36480 nonzeros.
Reduced MIP has 3840 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.06 sec. (28.28 ticks)
Probing time = 0.09 sec. (6.95 ticks)
Tried aggregator 1 time.
Detecting symmetries...
MIP Presolve eliminated 3150 rows and 225 columns.
Reduced MIP has 6810 rows, 4290 columns, and 27030 nonzeros.
Reduced MIP has 3840 binaries, 0 generals, 0 SOSs, and 

In [5]:
mutable struct Master
    model::Model
    w::Vector{VariableRef} # variables
    demand_constr::Vector{<:ConstraintRef} # constraints
    SRI::Vector{<:ConstraintRef}   # constraints
    routes::Vector{Vector{Float64}}
    # cost::Vector{Vector{Int64}}
    R::Int64 # number of routes
end

In [6]:
function Master(data)
    routes = Vector{Vector{Float64}}(undef,0)
    # cost = Vector{Vector{Int64}}(undef,0)
    n = data.Vc
    d = data.d
    cd = d[:, end]
    K=n
    S = [3, 5, 7, 9, 11, 13, 15]
    k = [2, 3, 4, 5, 6, 7, 8]
            
    # statement 1: construct initial naive configurations 
    for i=1:n
        a = zeros(Int, n)
        a[i] = Int(floor(1 / ones(Int,n)[i]))
        push!(routes, a)
    end
    
    
    R = length(routes) # statement 2: Let P denote the number of initial routes
    
    c = zeros(Int, R)
    for r in 1:R
        c[r] = 2*cd[r]
        # push!(cost, c)
    end

    # define empty model
    model = Model(HiGHS.Optimizer)  # Using HiGHS optimizer instead of CPLEX to avoid CPLEX 1217 error
    
    # mute the output
    set_silent(model)

    # statement 3: add variable `w`, objective function, and constraint named 'demand_constr' to `model`
    
    @variable(model, w[1:R]>=0)
    @objective(model, Min, sum(c[r] * w[r] for r in 1:R))
    @constraint(model, demand_constr[i in 1:n],  routes[i]' * w >= 1)
    @constraint(model, SRI[s in 1:length(S)], sum(floor((1/k[s])*sum(routes[i][r] for i in 1:S[s]))*w[r] for r in 1:R) <= floor(length(S[s])/k[s]))

    # @constraint(model, VI,  sum(routes[i]/2 for i in 1:3) * w <= Int(floor(3/2)))
    # @constraint(model, VI, sum(`w[r] .- sum(routes[i] for i in 1:3)` for r in 1:R) <= 1)
    
    return Master(model, w, demand_constr, SRI, routes, R)
end

Master

In [7]:
mmip = Master(data)
@show mmip.model
@show mmip.routes
@show mmip.w;

mmip.model = A JuMP Model
Minimization problem with:
Variables: 15
Objective function type: AffExpr
`AffExpr`-in-`MathOptInterface.GreaterThan{Float64}`: 15 constraints
`AffExpr`-in-`MathOptInterface.LessThan{Float64}`: 7 constraints
`VariableRef`-in-`MathOptInterface.GreaterThan{Float64}`: 15 constraints
Model mode: AUTOMATIC
CachingOptimizer state: EMPTY_OPTIMIZER
Solver name: HiGHS
Names registered in the model: SRI, demand_constr, w
mmip.routes = [[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0,

In [8]:
optimize!(mmip.model)
solution_summary(mmip.model)

* Solver : HiGHS

* Status
  Result count       : 1
  Termination status : OPTIMAL
  Message from the solver:
  "kHighsModelStatusOptimal"

* Candidate solution (result #1)
  Primal status      : FEASIBLE_POINT
  Dual status        : FEASIBLE_POINT
  Objective value    : 4.46000e+02
  Objective bound    : 0.00000e+00
  Relative gap       : Inf
  Dual objective value : 4.46000e+02

* Work counters
  Solve time (sec)   : 0.00000e+00
  Simplex iterations : 0
  Barrier iterations : 0
  Node count         : -1


In [9]:
mutable struct Sub
    model::Model
    a::Vector{VariableRef} # variables
    x::Matrix{VariableRef}
    N::Vector{VariableRef}
end

function Sub(data)

    d = data.d
    q = data.q
    P = data.P
    Vc = data.Vc
    G = zeros(Int,Vc)
    for i in 1:Vc
        G[i] = sum(q[i, p] for p in 1:P)
    end
    n=Vc
    L = data.L
    K = n
    Qp = data.Qp
    Qmax = sum(Qp)

    # initial dummy dual values
    π̂ = zeros(Float64, n)
    Iπ̂   = zeros(Float64, 7)
    model = Model(HiGHS.Optimizer) # Using HiGHS optimizer instead of CPLEX to avoid CPLEX 1217 error

    set_silent(model)
    
    # statement 1: add constraints and objective to `model`
    
    @variable(model, x[1:n+1,1:n+1], Bin)
    @variable(model, y[1:n+1], Bin)
    @variable(model, z[1:n, 1:P], Bin)
    @variable(model, u[1:n, 1:P] >=0)
    @variable(model, a[1:n], Bin )
    @variable(model, N[1:7])

    # @constraint(model, [i in 1:Vc], sum(y[i]) == 1)
    # @constraint(model, sum(y[Vc+1]) <= K)
    @constraint(model, [j in 1:Vc+1], sum(x[i, j] for i in 1:(Vc+1) if i!=j) == y[j])
    @constraint(model, [i in 1:Vc+1], sum(x[i, j] for j in 1:(Vc+1) if j!=i) == y[i])
    @constraint(model, sum(G[i] * y[i] for i in 1:Vc) <= Qmax)
    @constraint(model, [j in 1:Vc, p in 1:P], z[j, p] <= sum(x[i, j] for i in 1:Vc+1 if i!=j))
    # @constraint(model, [j in 1:Vc, p in 1:P], z[j, p] == 1)
    @constraint(model, [p in 1:P], sum(z[j, p] * q[j, p] for j in 1:Vc) <= K * Qp[p])
    @constraint(model,  sum(d[i, j] * x[i, j] for i in 1:(Vc+1) for j in 1:(Vc+1) if i != j) <= L)
    @constraint(model, [i in 1:Vc, j in 1:Vc, j!=i, k in 1:K, p in 1:P], u[i, p] - u[j, p] + Qp[p] * x[i, j] <= Qp[p] - q[i, p])
    @constraint(model, [i in 1:Vc, p in 1:P], q[i, p] - u[i, p] <= 0)
    @constraint(model, [i in 1:Vc, k in 1:K, p in 1:P], Qp[p] - u[i, p] >= 0)
    @constraint(model, [s in 1:7], N[s] == 1)
    for i in 1:n
        @constraint(model, a[i] - sum(x[i, j] for j in 1:n+1) == 0)
    end
        
    @objective(model, Min, sum(d[i, j] * x[i, j] for i in 1:(Vc+1) for j in 1:(Vc+1)) + sum(N[s]*Iπ̂[s] for s in 1:7)-sum(π̂[i] * a[i] for i in 1:n))
    
    
    return Sub(model, a, x, N)
end

Sub

In [10]:
# generate the subproblem for CG
sub = Sub(data)

@show sub.model
@show sub.a;
@show sub.x;

sub.model = A JuMP Model
Minimization problem with:
Variables: 384
Objective function type: AffExpr
`AffExpr`-in-`MathOptInterface.EqualTo{Float64}`: 54 constraints
`AffExpr`-in-`MathOptInterface.GreaterThan{Float64}`: 675 constraints
`AffExpr`-in-`MathOptInterface.LessThan{Float64}`: 10220 constraints
`VariableRef`-in-`MathOptInterface.GreaterThan{Float64}`: 45 constraints
`VariableRef`-in-`MathOptInterface.ZeroOne`: 332 constraints
Model mode: AUTOMATIC
CachingOptimizer state: EMPTY_OPTIMIZER
Solver name: HiGHS
Names registered in the model: N, a, u, x, y, z
sub.a = VariableRef[a[1], a[2], a[3], a[4], a[5], a[6], a[7], a[8], a[9], a[10], a[11], a[12], a[13], a[14], a[15]]
sub.x = VariableRef[x[1,1] x[1,2] x[1,3] x[1,4] x[1,5] x[1,6] x[1,7] x[1,8] x[1,9] x[1,10] x[1,11] x[1,12] x[1,13] x[1,14] x[1,15] x[1,16]; x[2,1] x[2,2] x[2,3] x[2,4] x[2,5] x[2,6] x[2,7] x[2,8] x[2,9] x[2,10] x[2,11] x[2,12] x[2,13] x[2,14] x[2,15] x[2,16]; x[3,1] x[3,2] x[3,3] x[3,4] x[3,5] x[3,6] x[3,7] x[3,8] x

In [ ]:
optimize!(sub.model)
solution_summary(sub.model)

In [11]:
function runCG(master::Master, sub::Sub, data::Data)
    
    while true
        
        # statemet1: solve the restricted master problem
        
        # set_time_limit_sec(master.model, 60.0)
        optimize!(master.model)
        
        println(value.(master.w))
        println(objective_value(master.model))
        
        π = dual.(master.demand_constr) # statemet 2: get dual variables associated with demand_constraint
        Iπ = dual.(master.SRI) # statemet 2: get dual variables associted with SRI
        # statemet 3: update the subproblem objective function using the dual info
        set_objective_coefficient.(sub.model, sub.N, Iπ)
        set_objective_coefficient.(sub.model, sub.a, -π)
        # statemet 4: solve the subproblem
        # set_time_limit_sec(sub.model, 60.0)
        optimize!(sub.model)
        
        # statemet 5: obtain the reduced cost
        reduced_cost = objective_value(sub.model)
        if reduced_cost < -1e-8
            # statemet 6: add the column with negative reduced cost to the master model and routes
            â = value.(sub.a)
            xx = value.(sub.x)
            mc = sum(d[i, j] * xx[i,j] for i in 1:Vc+1, j in 1:Vc+1)
            push!(master.routes, â)
            push!(master.w, @variable(master.model, lower_bound = 0))
            set_objective_coefficient(master.model, master.w[end], mc)
            set_normalized_coefficient.(master.demand_constr, master.w[end], â)
            println("Found new route. Total routes = $(length(master.routes))")
        else
            break
            println("terminate")
        end
    end
end


runCG (generic function with 1 method)

In [12]:
# generate the master problem for CG
master = Master(data)
sub = Sub(data)

tic = time()
runCG(master, sub, data)
toc = time()

solution_time = toc-tic
println("solution time: ", solution_time)

objective_value(master.model)

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
446.0
Found new route. Total routes = 16
[0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, -0.0, 1.0, 1.0, -0.0, 1.0]
275.0
Found new route. Total routes = 17
[0.0, 1.0, 0.0, 2.9753977059954195e-14, 0.0, 1.0, -0.0, 1.0, 0.0, 1.9539925233402755e-14, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.9999999999999702]
249.99999999999994
Found new route. Total routes = 18
[-0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0, -1.0436096431476471e-14, 0.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.9999999999999705, 2.9531932455029164e-14]
249.99999999999952
Found new route. Total routes = 19
[0.0, 1.0, 0.49999999999998535, 0.0, -0.0, 1.0, 0.0, -0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.5000000000000001, 0.4999999999999999, 0.5000000000000001, 0.9999999999999999]
214.99999999999966
Found new route. Total routes = 20
[0.0, 1.0, -1.4876988529977098e-14, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.4999999999999999, 0.5, 0.0, 0.5000000000000001, 0.5000

151.53170731707309
Found new route. Total routes = 39
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.07165437302423575, 0.15489989462592196, 0.14436248682824054, 0.0, 0.0, 0.0, 0.0, 0.0, 0.525816649104321, 0.42992623814541536, 0.04425711275026357, 0.0, 0.41728134878819834, 0.10642781875658582, 0.0, 0.31928345626975696, 0.17913593256058977, 0.0, 0.1338250790305588, 0.08324552160168631, 0.19810326659641736, 0.025289778714435218, 0.09378292939936796]
151.14436248682802
Found new route. Total routes = 40
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.10526315789473702, 0.05263157894736967, 0.0, 0.0, 0.0, 0.0, 0.0, 0.47368421052631665, 0.4210526315789471, 0.0, 0.0, 0.3684210526315786, 0.105263157894737, 0.0, 0.42105263157894635, 0.1578947368421067, 0.0, 0.10526315789473761, 0.10526315789473711, 0.2631578947368409, 0.10526315789473567, 0.15789473684210442, 0.10526315789473624]
149.9473684210526
Found new route. Tota

143.5490196078431
Found new route. Total routes = 55
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.17127071823204387, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.4475138121546965, 0.02762430939226572, 0.15469613259668522, 0.0, 0.0, 0.35911602209944804, 0.4198895027624312, 0.0, 0.34806629834254116, 0.13812154696132625, 0.0, 0.27624309392265156, 0.0, 0.21546961325966824, 0.06077348066298365, 0.1381215469613253, 0.05524861878453021, 0.06077348066298313, 0.0773480662983426]
143.4088397790055
Found new route. Total routes = 56
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.12280701754385841, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.47368421052631643, 0.08771929824561592, 0.1052631578947354, 0.0, 0.0, 0.2631578947368411, 0.3859649122807018, 0.0, 0.36842105263157965, 0.0877192982456126, 0.0, 0.17543859649122714, 0

142.0

In [13]:
master.routes

62-element Vector{Vector{Float64}}:
 [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0

In [ ]:
println(value.(master.w))